# RAG Step 3: Retrieval and Similarity Search
Given a question, fetch the most relevant chunks.

In [ ]:
!pip install -q sentence-transformers chromadb

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
col = chromadb.PersistentClient(path="./chroma_store").get_collection("rag_demo")

def retrieve(question, k=3, where=None):
    q = model.encode([question], normalize_embeddings=True).tolist()
    res = col.query(query_embeddings=q, n_results=k, where=where)
    return [{"text": d, "source": m["source"], "score": 1 - dist}
            for d, m, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])]

## 1. Top-k retrieval

In [ ]:
for r in retrieve("Why use RAG instead of fine-tuning?"):
    print(f"{r['score']:.3f} [{r['source']}] {r['text'][:90]}")

## 2. Metadata filtering

In [ ]:
for r in retrieve("How is similarity measured?", k=2, where={"source": "vector_db.txt"}):
    print(f"{r['score']:.3f} {r['text'][:90]}")

## 3. Score threshold
Drop weak matches so the model is not given irrelevant context.

In [ ]:
def retrieve_filtered(question, k=5, min_score=0.3):
    return [r for r in retrieve(question, k) if r["score"] >= min_score]

print(len(retrieve_filtered("What is the capital of France?")), "chunks pass the threshold")

## 4. Maximal Marginal Relevance (diverse results)

In [ ]:
import numpy as np
def mmr(question, k=3, fetch=8, lam=0.6):
    cands = retrieve(question, fetch)
    qv = model.encode([question], normalize_embeddings=True)[0]
    cv = model.encode([c["text"] for c in cands], normalize_embeddings=True)
    chosen = []
    while len(chosen) < min(k, len(cands)):
        best, best_s = None, -1e9
        for i in range(len(cands)):
            if i in chosen: continue
            rel = cv[i] @ qv
            red = max((cv[i] @ cv[j] for j in chosen), default=0)
            s = lam * rel - (1 - lam) * red
            if s > best_s: best, best_s = i, s
        chosen.append(best)
    return [cands[i] for i in chosen]

for r in mmr("Tell me about RAG and vector search"):
    print(r["source"], "|", r["text"][:70])

## Takeaways
Retrieval quality caps answer quality. Use thresholds and MMR to reduce noise and redundancy.